# Lab 01: MCP Architecture Fundamentals

**Lab 01 Solution: MCP Architecture — Protocol Fundamentals**

Understand the Model Context Protocol architecture: Hosts, Clients, Servers,
Transports, and the three core primitives (Tools, Resources, Prompts).

No external packages required — standard library only.

In [ ]:
import os
import json
import shutil
from dataclasses import dataclass, field, asdict
from typing import List

WORKDIR = "/tmp/aidev-lab-13-01"

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)

score = 0
total = 0

## Step 1: MCP Architecture Components

The Model Context Protocol defines four key roles:

| Component | Description |
|-----------|-------------|
| Host | The application (IDE, chatbot) running the LLM |
| Client | Protocol connector inside the host (1:1 w/ server) |
| Server | Exposes tools/resources/prompts over MCP |
| Transport | Communication layer (stdio, HTTP+SSE) |

In [ ]:
print("  Host (e.g. Claude Desktop)")
print("    └── MCP Client  ──transport──▶  MCP Server  ──▶  Data/APIs")

## Step 2: The Three MCP Primitives

| Primitive | Controlled By | Purpose | Analogy |
|-----------|--------------|---------|----------|
| Tools | Model (LLM) | Actions the model can invoke (search, compute) | POST endpoint |
| Resources | Application | Data the app can read (files, DB rows, configs) | GET endpoint |
| Prompts | User | Templates the user selects (/review, /fix) | Slash command |

## Step 3: MCP vs Direct Integration (N*M vs N+M)

**Without MCP:** Each agent needs a custom integration per tool
- Integrations = N agents x M tools = N * M

**With MCP:** Each agent connects via MCP; each tool exposes MCP
- Integrations = N agents + M tools = N + M

**Example:** 3 agents, 4 tools
- Without MCP: 3 * 4 = 12 integrations
- With MCP: 3 + 4 = 7 integrations

## TODO 1 Solution: Map Primitives to Controllers

Map each MCP primitive to who controls it.

In [ ]:
primitive_controllers = {
    "tools":     "model",
    "resources": "application",
    "prompts":   "user",
}

In [ ]:
total += 1
expected_controllers = {"tools": "model", "resources": "application", "prompts": "user"}
if primitive_controllers == expected_controllers:
    score += 1
    print("[PASS] Primitive-to-controller mapping is correct")
else:
    print("[FAIL] Expected:", expected_controllers)
    print("       Got:     ", primitive_controllers)

## TODO 2 Solution: Calculate Integration Counts

Calculate N*M vs N+M integration counts.

In [ ]:
N = 5   # number of agents
M = 10  # number of tools

without_mcp = N * M  # 50
with_mcp    = N + M  # 15

In [ ]:
total += 1
if without_mcp == 50 and with_mcp == 15:
    score += 1
    print(f"[PASS] Without MCP = {without_mcp}, With MCP = {with_mcp}")
else:
    print(f"[FAIL] Expected without_mcp=50, with_mcp=15")
    print(f"       Got without_mcp={without_mcp}, with_mcp={with_mcp}")

## TODO 3 Solution: Build an MCPServerInfo Data Structure

Build an MCPServerInfo dataclass instance.

In [ ]:
@dataclass
class MCPServerInfo:
    """Describes an MCP server's capabilities."""
    name: str = ""
    version: str = ""
    tools: List[str] = field(default_factory=list)
    resources: List[str] = field(default_factory=list)
    prompts: List[str] = field(default_factory=list)

server_info = MCPServerInfo(
    name="code-assistant",
    version="1.0.0",
    tools=["search_code", "run_linter", "analyze_deps"],
    resources=["file://project", "config://settings"],
    prompts=["code_review", "explain_function"],
)

In [ ]:
total += 1
info_dict = asdict(server_info)
checks = [
    info_dict["name"] == "code-assistant",
    info_dict["version"] == "1.0.0",
    info_dict["tools"] == ["search_code", "run_linter", "analyze_deps"],
    info_dict["resources"] == ["file://project", "config://settings"],
    info_dict["prompts"] == ["code_review", "explain_function"],
]
if all(checks):
    score += 1
    print("[PASS] MCPServerInfo is correct")
    out_path = os.path.join(WORKDIR, "server_info.json")
    with open(out_path, "w") as f:
        json.dump(info_dict, f, indent=2)
    print(f"       Saved to {out_path}")
else:
    print("[FAIL] MCPServerInfo does not match expected values")
    print("       Got:", json.dumps(info_dict, indent=2))

## TODO 4 Solution: Classify JSON-RPC Methods by Primitive

MCP uses these JSON-RPC methods:

| Method | Description |
|--------|-------------|
| tools/list | List available tools |
| tools/call | Execute a tool |
| resources/list | List available resources |
| resources/read | Read a resource's content |
| prompts/list | List available prompt templates |
| prompts/get | Retrieve a prompt template |
| initialize | Handshake with capabilities exchange |

In [ ]:
method_classification = {
    "tools/list":     "tool",
    "tools/call":     "tool",
    "resources/list": "resource",
    "resources/read": "resource",
    "prompts/list":   "prompt",
    "prompts/get":    "prompt",
    "initialize":     "lifecycle",
}

In [ ]:
total += 1
expected_classification = {
    "tools/list":     "tool",
    "tools/call":     "tool",
    "resources/list": "resource",
    "resources/read": "resource",
    "prompts/list":   "prompt",
    "prompts/get":    "prompt",
    "initialize":     "lifecycle",
}
if method_classification == expected_classification:
    score += 1
    print("[PASS] Method classification is correct")
    out_path = os.path.join(WORKDIR, "method_classification.json")
    with open(out_path, "w") as f:
        json.dump(method_classification, f, indent=2)
    print(f"       Saved to {out_path}")
else:
    print("[FAIL] Expected:", json.dumps(expected_classification, indent=2))
    print("       Got:     ", json.dumps(method_classification, indent=2))

## Summary

In [ ]:
print(f"Lab 01 Score: {score}/{total}")